## Demo 3: Geodata

Bicycle routes through Berlin as GPX files, and the borders of every country in the world as GeoJSON.

The first two demos moved rows whose types a table could describe: numbers, dates, strings. This one moves *geometry* - and no driver in this repository can hand a geometry to a database directly. Everything goes through **WKT**, "well known text", which every one of the three systems can read and write.

### What data do we have?

In [ ]:
import os

data_path = r"..\data\geodata"

for entry in sorted(os.listdir(data_path)):
    full = os.path.join(data_path, entry)
    if os.path.isdir(full):
        print(f"{entry + '/':40} {len(os.listdir(full))} files")
    else:
        print(f"{entry:40} {os.path.getsize(full) / 1024 / 1024:6.1f} MB")

In [ ]:
for file in sorted(os.listdir(os.path.join(data_path, "radrouten-berlin")))[:6]:
    print(file)

### A GPX file is XML, with three kinds of thing in it

A **track** is a recorded line, split into segments. A **route** is a planned line. A **waypoint** is a single point. The Berlin cycle routes are tracks; the travel guide file has tracks and waypoints.

In [ ]:
with open(r"..\data\geodata\radrouten-berlin\europaradweg_r1_ost.gpx", encoding="utf-8") as file:
    lines = file.readlines()

print("".join(lines[:8]))
print("...")
print("".join(lines[-4:]))

#### And here is the first thing that only bites in Python

PowerShell reads `([xml]$content).gpx` and then `$gpx.trk`, and never has to know which XML namespace the document declares. `ElementTree` does: a `<trk>` arrives as `{http://www.topografix.com/GPX/1/1}trk`, and `findall("trk")` finds nothing at all.

That would be a small annoyance if all the files agreed. They do not.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

namespaces = Counter()

for file in sorted(Path(data_path).glob("radrouten-berlin/*.gpx")):
    root = ET.parse(file).getroot()
    namespaces[root.tag.split("}")[0].strip("{")] += 1

for namespace, count in namespaces.items():
    print(f"{count:3} files  {namespace}")

Two namespaces, in one download, from one source.

So pinning the namespace to the version you happened to test against would return **zero rows for six of the twenty files** - no error, no warning, and a perfectly plausible result from the other fourteen.

`import_gpx_file` matches with `{*}` instead, which accepts any namespace and is the closest thing Python has to `$gpx.trk`.

### From GPX to WKT

`import_gpx_file` walks the tracks, routes and waypoints of every file matching a pattern, and turns each one into a WKT string: a `LINESTRING` for a single segment, a `MULTILINESTRING` for several, a `POINT` for a waypoint.

The path is a *pattern*, not a directory - the same as the sibling function, and for the same reason demo 1 learned the hard way.

In [ ]:
from import_gpx_file import import_gpx_file

tours = import_gpx_file(r"..\data\geodata\radrouten-berlin\*.gpx")

tours

In [ ]:
print(tours["wkt"][0][:300], "...")
print()
print("longest geometry:", tours["wkt"].map(len).max(), "characters")

In [ ]:
guide = import_gpx_file(r"..\data\geodata\michael-mueller-verlag-berlin.gpx")

guide["type"].value_counts()

In [ ]:
guide[guide["type"] == "Waypoint"].head()

One thing here is shorter in Python than in PowerShell, which does not happen often in this port. A GPX name is frequently a CDATA section, and the sibling has to test `$name.'#cdata-section'` and fall back to it. `ElementTree` hands over the text of a CDATA section like any other text, so there is no special case at all.

In [ ]:
import pandas as pd

data = pd.concat([tours, guide], ignore_index=True)

print(len(data), "rows")
data["type"].value_counts()

### Into SQL Server

The table has a `GEOMETRY` column. We do not send a geometry - we send the WKT string and let the database build the geometry inside the `VALUES` clause.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

# The counterpart of Import-Module PSFramework in the sibling: the bulk-load progress
# on screen, everything in demo.log, and every message kept so it can be queried later.
from configure_logging import configure_logging

messages = configure_logging()

from connect_sql_instance import connect_sql_instance
from invoke_sql_query import invoke_sql_query

sql_connection = connect_sql_instance(
    instance="127.0.0.1",
    database="Geodata",
    username="Geodata",
    password="Passw0rd!"
)

In [ ]:
invoke_sql_query(
    connection=sql_connection,
    query="CREATE TABLE dbo.berlin_tours (type VARCHAR(10), name VARCHAR(250), geometry GEOMETRY)"
)

`geometry::STGeomFromText(@wkt, 4326)` builds the geometry from the text, and `.MakeValid()` repairs a line that crosses itself. 4326 is WGS84 - plain latitude and longitude, which is what a GPS records.

That statement also found a bug. `invoke_sql_query` rewrites `@name` and `:name` into the `?` placeholders that pyodbc wants, and the `::` in `geometry::STGeomFromText` looked exactly like a named parameter to that regular expression. It failed with `KeyError: "Named parameter 'STGeomFromText' not provided"` - the very first statement of this demo, in a function two other demos had been using for weeks. Both databases use `::` as an operator, so all three `invoke_*_query` functions now refuse to read a doubled colon as a parameter.

In [ ]:
insert_query = "INSERT INTO dbo.berlin_tours VALUES (@type, @name, geometry::STGeomFromText(@wkt, 4326).MakeValid())"

for row in data.itertuples(index=False):
    invoke_sql_query(
        connection=sql_connection,
        query=insert_query,
        parameter_values={"type": row.type, "name": row.name, "wkt": row.wkt},
        enable_exception=True
    )

invoke_sql_query(
    connection=sql_connection,
    query="SELECT type, COUNT(*) AS rows_written FROM dbo.berlin_tours GROUP BY type"
)

### Getting it back out again

Now the other direction, and it does not work the obvious way.

In [ ]:
# No enable_exception, so this prints the error and returns None instead of raising

invoke_sql_query(connection=sql_connection, query="SELECT TOP 2 * FROM dbo.berlin_tours")

`ODBC SQL type -151 is not yet supported` - pyodbc has no idea what to do with a `GEOMETRY` column.

The sibling hits the same wall with a different message, `DataReader.GetFieldType(2) returned null`. Both drivers refuse; neither can represent the type.

So we ask the database to convert it on the way out, exactly as we asked it to convert on the way in.

In [ ]:
sql_export = invoke_sql_query(
    connection=sql_connection,
    query="SELECT type, name, geometry.STAsText() AS wkt FROM dbo.berlin_tours"
)

sql_export

### Into PostgreSQL, from the data we just read out of SQL Server

Same shape, different function names, and PostGIS spells the conversion `ST_GeomFromText` instead of `geometry::STGeomFromText`.

In [ ]:
from connect_pg_instance import connect_pg_instance
from invoke_pg_query import invoke_pg_query

pg_connection = connect_pg_instance(
    instance="127.0.0.1",
    database="geodata",
    username="geodata",
    password="Passw0rd!"
)

invoke_pg_query(
    connection=pg_connection,
    query="CREATE TABLE berlin_tours (type VARCHAR(10), name VARCHAR(250), geometry GEOMETRY)"
)

In [ ]:
for row in sql_export.itertuples(index=False):
    invoke_pg_query(
        connection=pg_connection,
        query="INSERT INTO berlin_tours VALUES (:type, :name, ST_MakeValid(ST_GeomFromText(:wkt, 4326)))",
        parameter_values={"type": row.type, "name": row.name, "wkt": row.wkt},
        enable_exception=True
    )

invoke_pg_query(connection=pg_connection, query="SELECT COUNT(*) FROM berlin_tours", as_type="single_value")

#### The same question, a different answer

In [ ]:
frame = invoke_pg_query(connection=pg_connection, query="SELECT * FROM berlin_tours LIMIT 2")

print("type of the geometry cell:", type(frame["geometry"][0]).__name__)
print()
print(frame["geometry"][0][:80], "...")

PostgreSQL does not refuse - it hands over a string. That string is hex EWKB, the internal binary format written out in hexadecimal, and it is not something you want on a slide.

Three systems, three answers to `SELECT *` on a geometry column:

| | |
| --- | --- |
| SQL Server | refuses: `ODBC SQL type -151 is not yet supported` |
| PostgreSQL | a `str` of hex EWKB, `0102000020E6100000…` |
| Oracle | an `SDO_GEOMETRY` object |

Only one of the three tells you that you asked the wrong question. Which is why the demo always asks for WKT.

#### One row at a time is the slow way

Every insert above was its own statement with its own round trip. The faster route is the one the StackExchange demo already used: bulk load the *text* into a staging table, then let one `INSERT ... SELECT` do all the conversions inside the database.

`write_pg_table` takes the DataFrame straight from SQL Server - the `wkt` column is just text, so `COPY` handles it with no geometry involved at all.

In [ ]:
from write_pg_table import write_pg_table

invoke_pg_query(
    connection=pg_connection,
    query="CREATE TABLE berlin_tours_import (type VARCHAR(10), name VARCHAR(250), wkt TEXT)",
    enable_exception=True
)

write_pg_table(connection=pg_connection, table="berlin_tours_import", data=sql_export)

invoke_pg_query(connection=pg_connection, query="TRUNCATE TABLE berlin_tours", enable_exception=True)
invoke_pg_query(
    connection=pg_connection,
    query="INSERT INTO berlin_tours SELECT type, name, ST_MakeValid(ST_GeomFromText(wkt, 4326)) FROM berlin_tours_import",
    enable_exception=True
)
invoke_pg_query(connection=pg_connection, query="DROP TABLE berlin_tours_import", enable_exception=True)

invoke_pg_query(connection=pg_connection, query="SELECT COUNT(*) FROM berlin_tours", as_type="single_value")

### Into Oracle

Oracle calls the type `SDO_GEOMETRY`, builds it with a constructor rather than a function, and repairs geometry with `SDO_UTIL.RECTIFY_GEOMETRY` rather than `MakeValid`. The shape of the demo does not change.

In [ ]:
from connect_ora_instance import connect_ora_instance
from invoke_ora_query import invoke_ora_query

ora_connection = connect_ora_instance(
    instance="127.0.0.1/XEPDB1",
    username="geodata",
    password="Passw0rd!"
)

invoke_ora_query(
    connection=ora_connection,
    query="CREATE TABLE berlin_tours (type VARCHAR2(10), name VARCHAR2(250), geometry SDO_GEOMETRY)"
)

In [ ]:
for row in sql_export.itertuples(index=False):
    invoke_ora_query(
        connection=ora_connection,
        query="INSERT INTO berlin_tours VALUES (:type, :name, SDO_UTIL.RECTIFY_GEOMETRY(SDO_GEOMETRY(:wkt, 4326), 0.01))",
        parameter_values={"type": row.type, "name": row.name, "wkt": row.wkt},
        enable_exception=True
    )

invoke_ora_query(
    connection=ora_connection,
    query='SELECT type, COUNT(*) AS rows_written FROM "BERLIN_TOURS" GROUP BY type'
)

A count rather than the geometry, and that is deliberate: reading these back as WKT is unreliable on Oracle, and the GeoJSON section below goes into why.

Then the staging table route again, which needs `write_ora_table` and a `CLOB` for the text - the longest of these geometries is 27070 characters, well past what a `VARCHAR2` would take.

In [ ]:
from write_ora_table import write_ora_table

invoke_ora_query(
    connection=ora_connection,
    query="CREATE TABLE berlin_tours_import (type VARCHAR2(10), name VARCHAR2(250), wkt CLOB)",
    enable_exception=True
)

write_ora_table(connection=ora_connection, table="berlin_tours_import", data=sql_export)

invoke_ora_query(connection=ora_connection, query='TRUNCATE TABLE "BERLIN_TOURS"', enable_exception=True)
invoke_ora_query(
    connection=ora_connection,
    query='INSERT INTO "BERLIN_TOURS" SELECT type, name, SDO_UTIL.RECTIFY_GEOMETRY(SDO_GEOMETRY(wkt, 4326), 0.01) FROM "BERLIN_TOURS_IMPORT"',
    enable_exception=True
)
invoke_ora_query(connection=ora_connection, query='DROP TABLE "BERLIN_TOURS_IMPORT"', enable_exception=True)

invoke_ora_query(connection=ora_connection, query='SELECT COUNT(*) FROM "BERLIN_TOURS"', as_type="single_value")

### Bonus: GeoJSON, and a limit worth knowing about

The country borders come as one GeoJSON document: a `FeatureCollection`, one feature per country, each with properties and a geometry.

In [ ]:
import json

with open(r"..\data\geodata\countries.geojson", encoding="utf-8") as file:
    geojson = json.load(file)

print("keys:      ", list(geojson.keys()))
print("crs:       ", geojson["crs"])   # CRS84 = WGS84 = EPSG:4326
print("features:  ", len(geojson["features"]))
print()
geojson["features"][0]["properties"]

In [ ]:
sizes = [len(json.dumps(feature["geometry"], separators=(",", ":"))) for feature in geojson["features"]]

print("geometry as JSON, in characters:")
print("  smallest", min(sizes))
print("  largest ", max(sizes))
print("  over 4000:", sum(1 for size in sizes if size > 4000), "of", len(sizes))

This time we do **not** go through WKT. PostgreSQL and Oracle can both read GeoJSON directly, with `ST_GeomFromGeoJSON` and `SDO_UTIL.FROM_GEOJSON`. SQL Server cannot - it only speaks WKT, so it sits this section out.

So the geometry goes to the database as the JSON string it already is.

In [ ]:
invoke_pg_query(
    connection=pg_connection,
    query="CREATE TABLE countries (name VARCHAR(50), iso CHAR(3), geometry GEOMETRY)"
)

for feature in geojson["features"]:
    invoke_pg_query(
        connection=pg_connection,
        query="INSERT INTO countries VALUES (:name, :iso, ST_MakeValid(ST_SetSRID(ST_GeomFromGeoJSON(:geometry), 4326)))",
        parameter_values={
            "name": feature["properties"].get("name"),
            "iso": feature["properties"].get("ISO3166-1-Alpha-3"),
            "geometry": json.dumps(feature["geometry"], separators=(",", ":"))
        },
        enable_exception=True
    )

invoke_pg_query(
    connection=pg_connection,
    query="SELECT name, iso, ST_NPoints(geometry) AS points FROM countries ORDER BY points DESC LIMIT 5"
)

#### The 4000 character limit

The same loop against Oracle failed for 71 of the 258 countries:

```
ORA-01461: can bind a LONG value only for insert into a LONG column
```

A parameter longer than 4000 characters has to be declared a `CLOB`, and 190 of these 258 geometries are. The sibling's `Invoke-OraQuery` has carried that guard all along:

```powershell
} elseif ($ParameterValues[$parameterName].Length -gt 4000) {
    $parameter.OracleDbType = 'CLOB'
}
```

`invoke_ora_query` did not, because nothing in the first two demos ever passed a long parameter. It does now.

What makes it worth a slide: this is the **opposite** of what the bulk path wants. In `write_ora_table`, declaring a `CLOB` column made an import thirty times slower, so it deliberately does not. Here, declaring it is the only way the value arrives at all. Same driver, same type - because a value going into a `CLOB` *column* and a value going into a *function argument* are not the same question.

In [ ]:
invoke_ora_query(
    connection=ora_connection,
    query="CREATE TABLE countries (name VARCHAR2(50), iso CHAR(3), geometry SDO_GEOMETRY)"
)

for feature in geojson["features"]:
    invoke_ora_query(
        connection=ora_connection,
        query="INSERT INTO countries VALUES (:name, :iso, SDO_UTIL.FROM_GEOJSON(:geometry))",
        parameter_values={
            "name": feature["properties"].get("name"),
            "iso": feature["properties"].get("ISO3166-1-Alpha-3"),
            "geometry": json.dumps(feature["geometry"], separators=(",", ":"))
        },
        enable_exception=True
    )

invoke_ora_query(connection=ora_connection, query='SELECT COUNT(*) FROM "COUNTRIES"', as_type="single_value")

#### And a dead end, left in on purpose

All 258 countries are in the table. Reading them back out as WKT is where Oracle stops cooperating.

In [ ]:
invoke_ora_query(
    connection=ora_connection,
    query='SELECT name, iso, SDO_UTIL.TO_WKTGEOMETRY(geometry) AS wkt FROM "COUNTRIES"'
)

`ORA-13199: wk buffer merge failure`, which the sibling records at this same line with a link to the Stack Overflow question about it.

One bad row takes down the whole statement, so it is worth asking which rows are bad.

In [ ]:
converted = []
failed = []

for (name,) in invoke_ora_query(connection=ora_connection, query='SELECT name FROM "COUNTRIES"', as_type="list"):
    wkt = invoke_ora_query(
        connection=ora_connection,
        query='SELECT SDO_UTIL.TO_WKTGEOMETRY(geometry) AS wkt FROM "COUNTRIES" WHERE name = :name',
        parameter_values={"name": name},
        as_type="single_value"
    )
    (converted if wkt is not None else failed).append(name)

print(f"converted {len(converted)}, failed {len(failed)}")
print()
print("failing:", failed[:10])

Around a fifth of them, and the interesting part is everything that does **not** explain it:

- **Not size.** Canada is the largest geometry in the file at 1.5 MB of JSON, and it converts fine.
- **Not validity.** `SDO_GEOM.VALIDATE_GEOMETRY_WITH_CONTEXT` returns the same `13367 [Element <1>] [Ring <1>]` for rows that convert and rows that do not.
- **Not a missing repair step.** Rectifying on the way out rescued 10 of 57 in one run. And the `BERLIN_TOURS` table above *is* rectified on the way in, row by row - 14 of its 49 geometries failed to convert back all the same.
- **Not even repeatable.** Four runs over identical data gave 59, 57, 49 and 49 failures - and the membership moves too. Oman and Uzbekistan converted in an earlier run and are in the failing list above.

That last point is the one to keep. This is not a property of particular countries that could be cleaned up in advance: it is `SDO_UTIL.TO_WKTGEOMETRY` giving different answers about the same stored geometry from one run to the next. Which is why the sibling links a Stack Overflow question here rather than a fix.

It stays in the notebook as a dead end, because a demo that shows only the parts that worked teaches the wrong lesson about how this kind of work actually goes.

And it means the WKT round trip is **not symmetrical on Oracle**: geometry goes in reliably and does not always come back out. On SQL Server and PostgreSQL it does.

### Key takeaways

* Most geodata formats can be turned into WKT, and WKT is what all three systems read and write.
* The conversion happens in the `VALUES` clause, on the way in, and in the `SELECT` list on the way out. The driver never sees a geometry.
* `SELECT *` on a geometry column gives three different answers on three systems, and only one of them is an error.
* For more than a handful of rows: bulk load the text into a staging table, then convert with one `INSERT ... SELECT`.
* GeoJSON goes into PostgreSQL and Oracle directly, and not into SQL Server at all.

### Cleanup

In [ ]:
for query in [
    "DROP TABLE dbo.berlin_tours"
]:
    invoke_sql_query(connection=sql_connection, query=query, enable_exception=True)

for query in [
    "DROP TABLE berlin_tours",
    "DROP TABLE countries"
]:
    invoke_pg_query(connection=pg_connection, query=query, enable_exception=True)

for query in [
    'DROP TABLE "BERLIN_TOURS"',
    'DROP TABLE "COUNTRIES"'
]:
    invoke_ora_query(connection=ora_connection, query=query, enable_exception=True)

sql_connection.close()
pg_connection.close()
ora_connection.close()